In [1]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q datasets evaluate rouge_score textstat sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 97.4 MB/s eta 0:00:00


In [2]:
import torch
import re
import pandas as pd
import textstat

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

import evaluate

In [3]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)


model.eval()

print("Model loaded successfully")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded successfully


In [4]:
def zero_shot_prompt(text):
    return f"""
Simplify the following text into plain language.

Return only the simplified sentence.

Text:
{text}

Simplified:
"""


def few_shot_prompt(text):
    return f"""
Simplify text into plain language.

Example:

Text:
The physician prescribed medication to alleviate symptoms.

Simplified:
The doctor gave medicine to reduce symptoms.


Text:
The committee postponed the implementation because of logistical problems.

Simplified:
The committee delayed the plan because of problems.


Text:
{text}

Simplified:
"""


def cot_prompt(text):
    return f"""
Think about how to simplify the following sentence.

Do not provide explanation.
Only output the simplified sentence.

Text:
{text}

Simplified:
"""

In [5]:
def generate(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)


    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )


    generated = outputs[0][inputs.input_ids.shape[1]:]


    text = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )


    for stop in ["Human:", "Assistant:", "User:"]:
        if stop in text:
            text = text.split(stop)[0]


    return text.strip()

In [6]:
asset = load_dataset(
    "facebook/asset",
    "simplification"
)

NUM_SAMPLES = 100

test_data = asset["test"].select(range(NUM_SAMPLES))

print("Total Samples:", len(test_data))

print("\nFirst Sample:")
print(test_data[0]["original"])

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

simplification/validation-00000-of-00001(…):   0%|          | 0.00/885k [00:00<?, ?B/s]

simplification/test-00000-of-00001.parqu(…):   0%|          | 0.00/170k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/359 [00:00<?, ? examples/s]

Total Samples: 100

First Sample:
One side of the armed conflicts is composed mainly of the Sudanese military and the Janjaweed, a Sudanese militia group recruited mostly from the Afro-Arab Abbala tribes of the northern Rizeigat region in Sudan.


In [7]:
zero_predictions = []
few_predictions = []
cot_predictions = []

sources = []
references = []

results = []


for i, sample in enumerate(test_data):

    source = sample["original"]
    reference = sample["simplifications"]


    zero_output = generate(
        zero_shot_prompt(source)
    )

    few_output = generate(
        few_shot_prompt(source)
    )

    cot_output = generate(
        cot_prompt(source)
    )


    sources.append(source)
    references.append(reference)

    zero_predictions.append(zero_output)
    few_predictions.append(few_output)
    cot_predictions.append(cot_output)


    results.append({
        "Original": source,
        "Reference": reference[0],
        "Zero Shot": zero_output,
        "Few Shot": few_output,
        "Chain of Thought": cot_output
    })


    print(f"{i+1}/100 completed")


results_df = pd.DataFrame(results)

print("Generation completed!")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


1/100 completed
2/100 completed
3/100 completed
4/100 completed
5/100 completed
6/100 completed
7/100 completed
8/100 completed
9/100 completed
10/100 completed
11/100 completed
12/100 completed
13/100 completed
14/100 completed
15/100 completed
16/100 completed
17/100 completed
18/100 completed
19/100 completed
20/100 completed
21/100 completed
22/100 completed
23/100 completed
24/100 completed
25/100 completed
26/100 completed
27/100 completed
28/100 completed
29/100 completed
30/100 completed
31/100 completed
32/100 completed
33/100 completed
34/100 completed
35/100 completed
36/100 completed
37/100 completed
38/100 completed
39/100 completed
40/100 completed
41/100 completed
42/100 completed
43/100 completed
44/100 completed
45/100 completed
46/100 completed
47/100 completed
48/100 completed
49/100 completed
50/100 completed
51/100 completed
52/100 completed
53/100 completed
54/100 completed
55/100 completed
56/100 completed
57/100 completed
58/100 completed
59/100 completed
60/100

In [8]:
results_df.head(10)

,Original,Reference,Zero Shot,Few Shot,Chain of Thought
0,One side of the armed conflicts is composed ma...,On one side of the conflicts are the Sudanese ...,"The Sudanese military and the Janjaweed, a mil...",One side of the war is made up of the Sudanese...,"The Sudanese military and the Janjaweed, a mil..."
1,"Jeddah is the principal gateway to Mecca, Isla...",Muslims are required to visit Mecca once in th...,"Jeddah is the main entrance to Mecca, the holi...","Jeddah is the main city to reach Mecca, Islam'...","Jeddah is the main entrance to Mecca, the holi..."
2,The Great Dark Spot is thought to represent a ...,The dark spot on Ne;tune may be a hole in the ...,The Great Dark Spot is believed to be a hole i...,The Great Dark Spot is believed to be a hole i...,Neptune's Great Dark Spot is believed to be a ...
3,"His next work, Saturday, follows an especially...",Next Saturday is a presentation of a successfu...,"On Saturday, a neurosurgeon has a particularly...","His next work, on Saturday, is after a very bu...",Saturday is the neurosurgeon's next work after...
4,"The tarantula, the trickster character, spun a...",The tarantula spun a black cord and attached i...,"The tarantula, acting like a trickster, spun a...","The tarantula, the trickster, spun a black str...","The tarantula, the trickster, pulled a black c..."
5,"There he died six weeks later, on 13 January 888.",He died there six weeks later on 13 January 888.,"He died on January 13, 888.","He died six weeks later on January 13, 888.\n\...","He died on 13 January 888, six weeks later."
6,They are culturally akin to the coastal people...,They are culturally similar to the coastal peo...,They are similar to the coastal people of Papu...,They are similar to the coastal people of Papu...,They are similar to the coastal people of Papu...
7,"Since 2000, the recipient of the Kate Greenawa...","Since 2000, the recipient of the Kate Greenawa...","Since 2000, the winner of the Kate Greenaway M...","Since 2000, the winner of the Kate Greenaway M...","Since 2000, the Kate Greenaway Medal winner ha..."
8,"Following the drummers are dancers, who often ...","After the drummers are dancers, who often play...",Dancers follow the drummers and sometimes play...,After the drummers come dancers who usually pl...,Dancers behind drummers often play sogo and ha...
9,The spacecraft consists of two main elements: ...,The spacecraft's two main elements are the NAS...,The spacecraft has two parts: the NASA Cassini...,The spacecraft has two parts: the NASA Cassini...,The Cassini orbiter and the Huygens probe make...


In [10]:
!pip install sacremoses
sari = evaluate.load("sari")
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 51.9 MB/s eta 0:00:00


In [11]:
references_text = []

for sample in test_data:
    references_text.append(
        sample["simplifications"]
    )

sources_text = [
    sample["original"]
    for sample in test_data
]

In [12]:
def evaluate_model(predictions):

    # SARI
    sari_score = sari.compute(
        sources=sources_text,
        predictions=predictions,
        references=references_text
    )


    # BLEU
    bleu_score = bleu.compute(
        predictions=predictions,
        references=[
            [ref[0]]
            for ref in references_text
        ]
    )


    # ROUGE-L
    rouge_score = rouge.compute(
        predictions=predictions,
        references=[
            ref[0]
            for ref in references_text
        ]
    )


    # FKGL
    fkgl_scores = []

    for text in predictions:
        fkgl_scores.append(
            textstat.flesch_kincaid_grade(text)
        )


    return {
        "SARI": sari_score["sari"],
        "BLEU": bleu_score["bleu"],
        "ROUGE-L": rouge_score["rougeL"],
        "FKGL": sum(fkgl_scores)/len(fkgl_scores)
    }

In [13]:
mistral_zero = evaluate_model(
    zero_predictions
)

mistral_few = evaluate_model(
    few_predictions
)

mistral_cot = evaluate_model(
    cot_predictions
)

In [14]:
mistral_results = pd.DataFrame([
    {
        "Model":"Mistral-7B-Instruct",
        "Prompt":"Zero Shot",
        **mistral_zero
    },
    {
        "Model":"Mistral-7B-Instruct",
        "Prompt":"Few Shot",
        **mistral_few
    },
    {
        "Model":"Mistral-7B-Instruct",
        "Prompt":"Chain of Thought",
        **mistral_cot
    }
])


mistral_results

,Model,Prompt,SARI,BLEU,ROUGE-L,FKGL
0,Mistral-7B-Instruct,Zero Shot,43.597674,0.217049,0.505757,9.444859
1,Mistral-7B-Instruct,Few Shot,44.957198,0.112748,0.296727,7.878894
2,Mistral-7B-Instruct,Chain of Thought,40.857454,0.255689,0.505997,9.570470
